In [ ]:
import os
import subprocess

def extract_frames_ffmpeg(video_path, output_folder, fps=6):
    """
    Uses FFmpeg to extract frames at given FPS quickly.

    Args:
        video_path (str): Path to the input video
        output_folder (str): Directory to save frames
        fps (int): Number of frames to extract per second
    """
    os.makedirs(output_folder, exist_ok=True)

    output_pattern = os.path.join(output_folder, "frame_%04d.jpg")

    # Build ffmpeg command
    command = [
        "ffmpeg",
        "-i", video_path,        # input video
        "-vf", f"fps={fps}",     # filter to extract frames at fixed fps
        output_pattern
    ]

    # Run the command
    subprocess.run(command, check=True)
    print(f"✅ Done! Frames saved to: {output_folder}")

# Example usage
if __name__ == "__main__":
    extract_frames_ffmpeg("/content/fake_video.mp4", "frames", fps=6)



✅ Done! Frames saved to: frames


In [ ]:
import os
from PIL import Image
from facenet_pytorch import MTCNN
import torchvision.transforms as transforms

# Initialize face aligner (image_size = 300 for ISTVT)
mtcnn = MTCNN(image_size=300, margin=20, post_process=True)

def align_faces_in_video(input_folder, output_folder):
    """
    Align all frames in a single video folder using MTCNN.
    Saves aligned faces to output_folder.
    """
    os.makedirs(output_folder, exist_ok=True)

    files = sorted([
        f for f in os.listdir(input_folder) if f.lower().endswith(('.jpg', '.png'))
    ])

    for file in files:
        img_path = os.path.join(input_folder, file)
        try:
            img = Image.open(img_path).convert('RGB')
            face = mtcnn(img)  # returns aligned face tensor or None
            if face is not None:
                save_path = os.path.join(output_folder, file)
                transforms.ToPILImage()(face).save(save_path)
            else:
                print(f"⚠️ No face found in {file}, skipping.")
        except Exception as e:
            print(f"❌ Failed to process {file}: {e}")

def align_all_video_folders(super_input_folder, super_output_folder):
    """
    Go through all video folders and align each frame inside.
    """
    os.makedirs(super_output_folder, exist_ok=True)

    subfolders = sorted([
        d for d in os.listdir(super_input_folder)
        if os.path.isdir(os.path.join(super_input_folder, d))
    ])

    for sub in subfolders:
        in_path = os.path.join(super_input_folder, sub)
        out_path = os.path.join(super_output_folder, sub)
        print(f"🔄 Aligning: {in_path}")
        align_faces_in_video(in_path, out_path)
        print(f"✅ Saved aligned faces to: {out_path}")


In [ ]:
if __name__ == "__main__":
    raw_frames_root = "/kaggle/working/raw_frames"           # Folder with video1/, video2/, etc.
    aligned_faces_root = "/kaggle/working/aligned_faces"     # Where aligned frames will go

    align_all_video_folders(raw_frames_root, aligned_faces_root)


NameError: name 'align_all_video_folders' is not defined

if __name__ == "__main__":
    dataset_path = "/kaggle/input/faceforensics"  # Replace with your folder path
    extract_frames_from_dataset(dataset_path, output_base_folder="frames_dataset", fps=6)

In [ ]:
from torchvision import transforms
from PIL import Image
import torch
import os

def load_aligned_faces_to_tensor(folder, image_size=299, max_frames=None):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)  # Xception expects [-1, 1]
    ])

    files = sorted([
        f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png'))
    ])

    if max_frames:
        files = files[:max_frames]

    tensor_list = []
    for file in files:
        img = Image.open(os.path.join(folder, file)).convert('RGB')
        tensor_list.append(transform(img))

    if not tensor_list:
        raise ValueError("No valid frames found in the folder.")

    return torch.stack(tensor_list)  # shape: [T, 3, 299, 299]
# Example usage
if __name__ == "__main__":
    frames_tensor=load_aligned_faces_to_tensor("frames")



In [ ]:
import torch
import torch.nn as nn
import timm

class XceptionEntryFlow(nn.Module):
    def __init__(self):
        super().__init__()
        full_model = timm.create_model('xception', pretrained=True, features_only=False)

        # full_model has a `_blocks` attribute which contains entry flow + middle flow + exit flow
        # Entry flow is first 3 blocks: block0 (64->128), block1 (128->256), block2 (256->728)
        self.stem = nn.Sequential(
            full_model.conv1,  # 3x3 conv
            full_model.bn1,
            full_model.act1,
            full_model.conv2,
            full_model.bn2,
            full_model.act2
        )
        self.block0 = full_model.block1  # 128
        self.block1 = full_model.block2  # 256
        self.block2 = full_model.block3  # 728

    def forward(self, x):
        x = self.stem(x)      # → [B, 64, 147, 147]
        x = self.block0(x)    # → [B, 128, 74, 74]
        x = self.block1(x)    # → [B, 256, 37, 37]
        x = self.block2(x)    # ✅ [B, 728, 19, 19]
        return x

# Test
'''if __name__ == "__main__":
    model = XceptionEntryFlow()
    features = model(frames_tensor)
    print(f"✅ Extracted Feature Shape: {features.shape}")  # Should be [8, 728, 19, 19]'''


'if __name__ == "__main__":\n    model = XceptionEntryFlow()\n    features = model(frames_tensor)\n    print(f"✅ Extracted Feature Shape: {features.shape}")  # Should be [8, 728, 19, 19]'

In [ ]:
def feature_maps_to_tokens(feature_maps):
    """
    Converts feature maps [T, C, H, W] → token sequence [T, N, C]
    (1×1 patches, no linear projection)

    Parameters:
        feature_maps (torch.Tensor): shape [T, C, H, W]

    Returns:
        tokens (torch.Tensor): shape [T, N, C] where N = H × W
    """
    T, C, H, W = feature_maps.shape

    # Reshape: [T, C, H, W] → [T, C, H*W] → [T, H*W, C]
    tokens = feature_maps.view(T, C, H * W).permute(0, 2, 1)  # shape: [T, N, C]

    return tokens


In [ ]:
if __name__ == "__main__":
    # 1. Extract features using entry flow
    entry_model = XceptionEntryFlow()
    dummy_frames = torch.randn(8, 3, 299, 299)  # [T=8, 3, H, W]
    with torch.no_grad():
        features = entry_model(frames_tensor)  # shape: [8, C, H, W]

    print("Feature map shape:", features.shape)

    # 2. Convert to patch tokens (1x1, no projection)
    tokens = feature_maps_to_tokens(features)  # shape: [T, N, C]
    print("Token shape:", tokens.shape)  # e.g., [8, 361, 728]


/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Feature map shape: torch.Size([182, 728, 19, 19])
Token shape: torch.Size([182, 361, 728])


In [ ]:
import torch
import torch.nn as nn

class ISTVT_TokenPreprocessor(nn.Module):
    def __init__(self, num_frames, num_patches, embed_dim):
        super().__init__()
        self.T = num_frames
        self.N = num_patches  # e.g., 19×19 = 361
        self.C = embed_dim    # e.g., 728

        # CLS tokens
        self.spatial_cls_token = nn.Parameter(torch.randn(1, 1, self.C))   # [1, 1, C]
        self.temporal_cls_token = nn.Parameter(torch.randn(1, 1, self.C))  # [1, 1, C]

        # Positional embedding
        self.pos_embedding = nn.Parameter(torch.randn(self.T + 1, self.N + 1, self.C))

    def forward(self, tokens):  # tokens: [T, N, C]
        T, N, C = tokens.shape
        assert T == self.T and N == self.N and C == self.C

        # 🔵 Add 1 spatial CLS token to each frame
        spatial_cls = self.spatial_cls_token.expand(T, 1, C)  # [T, 1, C]
        tokens_with_cls = torch.cat([spatial_cls, tokens], dim=1)  # [T, N+1, C]

        # 🔴 Add temporal CLS token on top
        temporal_cls = self.temporal_cls_token.expand(1, N + 1, C)  # [1, N+1, C]
        full_tokens = torch.cat([temporal_cls, tokens_with_cls], dim=0)  # [T+1, N+1, C]

        # ➕ Add positional embedding
        full_tokens = full_tokens + self.pos_embedding  # [T+1, N+1, C]

        return full_tokens


In [ ]:
if __name__ == "__main__":

    T, N, C = tokens.shape
    preprocessor = ISTVT_TokenPreprocessor(num_frames=T, num_patches=N, embed_dim=C)
    out = preprocessor(tokens)

    print("Output shape:", out.shape)  # should be [T+1, N+1, C] => [9, 362, 728]

Output shape: torch.Size([183, 362, 728])


In [ ]:
TRANFORMER BLOCK

In [ ]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv_proj = nn.Linear(dim, dim * 3)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, q_input, k_input, v_input):
        B, N, C = q_input.shape
        qkv = self.qkv_proj(q_input).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = attn_scores.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

class ISTVTBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.temporal_attn = MultiHeadAttention(dim, num_heads)
        self.spatial_attn = MultiHeadAttention(dim, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)

    def forward(self, x):  # x: (B, T+1, HW+1, C)
        B, T, S, C = x.shape
        x_sub = torch.cat([x[:, :2], x[:, 2:] - x[:, 1:-1]], dim=1)

        # Temporal Attention
        temp_qk = self.norm1(x_sub).view(B, T * S, C)
        temp_v = x.view(B, T * S, C)
        temp_out = self.temporal_attn(temp_qk, temp_qk, temp_v)
        temp_out = temp_out.view(B, T, S, C)
        temp_out = temp_out + x

        # Spatial Attention
        spat_in = self.norm2(temp_out).view(B * T, S, C)
        spat_out = self.spatial_attn(spat_in, spat_in, spat_in)
        spat_out = spat_out + spat_in
        spat_out = spat_out.view(B, T, S, C)

        # Feed Forward
        ffn_in = self.norm3(spat_out)
        ffn_out = self.ffn(ffn_in)
        return ffn_out + spat_out

class ISTVTTransformer(nn.Module):
    def __init__(self, dim, num_heads, depth=6):  # M = 6 by default
        super().__init__()
        self.blocks = nn.ModuleList([ISTVTBlock(dim, num_heads) for _ in range(depth)])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os


class ISTVT_TokenPreprocessor(nn.Module):
    def __init__(self, num_frames, num_patches, embed_dim):
        super().__init__()
        self.T = num_frames
        self.N = num_patches  # H x W
        self.C = embed_dim

        self.spatial_cls_token = nn.Parameter(torch.randn(1, 1, self.C))
        self.temporal_cls_token = nn.Parameter(torch.randn(1, 1, self.C))
        self.positional_embedding = nn.Parameter(torch.randn(self.T + 1, self.N + 1, self.C))

    def forward(self, patch_tokens):
        T, N, C = patch_tokens.shape
        assert T == self.T and N == self.N and C == self.C

        spatial_cls = self.spatial_cls_token.expand(T, -1, -1)  # [T, 1, C]
        tokens_with_spatial_cls = torch.cat([spatial_cls, patch_tokens], dim=1)  # [T, N+1, C]

        temporal_cls = self.temporal_cls_token.expand(1, self.N + 1, -1)  # [1, N+1, C]
        full_sequence = torch.cat([temporal_cls, tokens_with_spatial_cls], dim=0)  # [T+1, N+1, C]

        return full_sequence + self.positional_embedding


class SpatialTemporalAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.spatial_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.temporal_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        x = x.transpose(0, 1)  # [N+1, T+1, C] for temporal
        temp_input = self.norm1(x)
        temp_out, _ = self.temporal_attn(temp_input, temp_input, temp_input)
        x = x + temp_out

        x = x.transpose(0, 1)  # [T+1, N+1, C] for spatial
        spa_input = self.norm2(x)
        spa_out, _ = self.spatial_attn(spa_input, spa_input, spa_input)
        x = x + spa_out

        x = x + self.mlp(x)
        return x


class ISTVT_Transformer(nn.Module):
    def __init__(self, num_layers, embed_dim, num_heads):
        super().__init__()
        self.blocks = nn.ModuleList([
            SpatialTemporalAttentionBlock(embed_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x


class PredictionHead(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, 1)
        )

    def forward(self, cls_token):
        cls_token = self.norm(cls_token)
        return self.mlp(cls_token).squeeze()


class ISTVT(nn.Module):
    def __init__(self, num_frames, num_patches, embed_dim, num_heads=8, num_layers=4):
        super().__init__()
        self.preprocessor = ISTVT_TokenPreprocessor(num_frames, num_patches, embed_dim)
        self.transformer = ISTVT_Transformer(num_layers, embed_dim, num_heads)
        self.prediction_head = PredictionHead(embed_dim)

    def forward(self, x):  # x: [T, N, C]
        x = self.preprocessor(x)  # [T+1, N+1, C]
        x = self.transformer(x)  # [T+1, N+1, C]
        cls_token = x[0, 0, :]  # Temporal classification token
        logits = self.prediction_head(cls_token)  # Output logits
        return logits


# Saving checkpoint after some batches

def save_checkpoint(model, optimizer, epoch, batch_idx, path="checkpoint.pt"):
    torch.save({
        'epoch': epoch,
        'batch_idx': batch_idx,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, path)


# Training loop snippet for saving every N batches

def train_one_epoch(model, dataloader, optimizer, criterion, epoch, save_every=10, save_dir="./checkpoints"):
    model.train()
    os.makedirs(save_dir, exist_ok=True)
    for batch_idx, (video_tensor, label) in enumerate(dataloader):
        optimizer.zero_grad()
        video_tensor = video_tensor.to(next(model.parameters()).device)
        label = label.to(next(model.parameters()).device).float()

        B, T, N, C = video_tensor.shape
        outputs = []
        for i in range(B):
            out = model(video_tensor[i])  # [T, N, C]
            outputs.append(out)
        output = torch.stack(outputs)  # [B]

        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        if (batch_idx + 1) % save_every == 0:
            checkpoint_path = os.path.join(save_dir, f"checkpoint_epoch{epoch}_batch{batch_idx+1}.pt")
            save_checkpoint(model, optimizer, epoch, batch_idx + 1, path=checkpoint_path)
            print(f"✅ Saved checkpoint: {checkpoint_path}")


if __name__ == "__main__":
    T, H, W = 8, 19, 19
    N = H * W
    C = 728

    dummy_tokens = torch.randn(T, N, C)
    model = ISTVT(num_frames=T, num_patches=N, embed_dim=C)
    output = model(dummy_tokens)
    print("✅ Output logit:", output.item())


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import timm

# === PARAMETERS ===
FRAMES_PER_CLIP = 6
IMAGE_SIZE = 299
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4  # clips per batch

# === TRANSFORM ===
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

# === YOUR CUSTOM XCEPTION ENTRY FLOW ===
class XceptionEntryFlow(nn.Module):
    def __init__(self):
        super().__init__()
        full_model = timm.create_model('xception', pretrained=True, features_only=False)

        self.stem = nn.Sequential(
            full_model.conv1,
            full_model.bn1,
            full_model.act1,
            full_model.conv2,
            full_model.bn2,
            full_model.act2
        )
        self.block0 = full_model.block1
        self.block1 = full_model.block2
        self.block2 = full_model.block3  # Output: [B, 728, 19, 19]

    def forward(self, x):
        x = self.stem(x)
        x = self.block0(x)
        x = self.block1(x)
        x = self.block2(x)
        return x  # [B, 728, 19, 19]

# === EXTRACT TOKEN PATCHES ===
def extract_feature_tokens(frames_tensor, model):
    B, T, C, H, W = frames_tensor.shape
    frames_tensor = frames_tensor.view(-1, C, H, W)  # [B*T, C, H, W]
    features = model(frames_tensor)  # [B*T, 728, 19, 19]
    features = features.view(B, T, 728, 19, 19)  # [B, T, C, H', W']
    return features.permute(0, 1, 3, 4, 2).reshape(B, T, -1, 728)  # [B, T, 361, 728]

# === MAIN CLIP PROCESSING WITH BATCHING + SAVING ===
def process_all_clips_batched(input_root, output_root, model):
    os.makedirs(output_root, exist_ok=True)
    clip_id = 0

    for label_dir, label in [('fake_align', 0), ('real_align', 1)]:
        label_input_path = os.path.join(input_root, label_dir)
        label_output_path = os.path.join(output_root, 'fake' if label == 0 else 'real')
        os.makedirs(label_output_path, exist_ok=True)

        if not os.path.exists(label_input_path):
            print(f"⚠️ Skipping missing folder: {label_input_path}")
            continue

        all_clips = []
        for video_folder in tqdm(os.listdir(label_input_path), desc=f"Loading {label_dir}"):
            folder_path = os.path.join(label_input_path, video_folder)
            if not os.path.isdir(folder_path):
                continue

            frames = sorted([
                f for f in os.listdir(folder_path)
                if f.lower().endswith(('.jpg', '.png'))
            ])
            usable = len(frames) - (len(frames) % FRAMES_PER_CLIP)

            for i in range(0, usable, FRAMES_PER_CLIP):
                clip_paths = [os.path.join(folder_path, frames[i + j]) for j in range(FRAMES_PER_CLIP)]
                try:
                    clip_tensor = torch.stack([
                        transform(Image.open(p).convert('RGB')) for p in clip_paths
                    ])  # [T, 3, H, W]
                    all_clips.append((clip_tensor, label))
                except:
                    print(f"❌ Skipping broken clip at {folder_path}")

                # === Process batch ===
                if len(all_clips) >= BATCH_SIZE:
                    save_batch_tokens(all_clips, model, label_output_path, clip_id)
                    clip_id += len(all_clips)
                    all_clips = []
                    torch.cuda.empty_cache()

        # Final leftover clips
        if all_clips:
            save_batch_tokens(all_clips, model, label_output_path, clip_id)
            clip_id += len(all_clips)
            all_clips = []
            torch.cuda.empty_cache()

    print(f"\n✅ All clips processed and saved to '{output_root}'")

# === TOKEN SAVE FUNCTION ===
def save_batch_tokens(batch_clips, model, save_dir, start_id):
    model.train()  # fine-tuning mode
    batch_tensor = torch.stack([clip[0] for clip in batch_clips]).to(DEVICE)  # [B, T, 3, H, W]
    labels = [clip[1] for clip in batch_clips]
    tokens = extract_feature_tokens(batch_tensor, model)  # [B, T, 361, 728]

    for i in range(len(tokens)):
        data = {
            'tokens': tokens[i].cpu(),  # [T, 361, 728]
            'label': labels[i]
        }
        save_path = os.path.join(save_dir, f"clip_{start_id + i:05d}.pt")
        torch.save(data, save_path)

# === SAVE FINE-TUNED MODEL ===
def save_finetuned_model(model, path="xception_entry_finetuned.pt"):
    torch.save(model.state_dict(), path)
    print(f"✅ Fine-tuned Xception weights saved to: {path}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    dataset_root = "/mnt/data"  # aligned data folder
    output_root = "./processed_clips"

    xception_model = XceptionEntryFlow().to(DEVICE)
    process_all_clips_batched(dataset_root, output_root, xception_model)
    save_finetuned_model(xception_model)



In [ ]:
if __name__ == "__main__":
    dataset_root = "/mnt/data"  # where your fake_align/real_align live
    output_root = "./processed_clips"  # where token+label .pt files will go

    xception_model = XceptionEntryFlow().to(DEVICE)  # GPU model
    process_all_clips_batched(dataset_root, output_root, xception_model)
    save_finetuned_model(xception_model)


In [ ]:
import os
import shutil

def create_merged_structure(source1, source2, destination):
    os.makedirs(destination, exist_ok=True)

    for src in [source1, source2]:
        folder_name = os.path.basename(src.rstrip('/'))
        dst_path = os.path.join(destination, folder_name)

        # Copy the entire folder
        if not os.path.exists(dst_path):
            shutil.copytree(src, dst_path)
        else:
            print(f"⚠️ Folder already exists: {dst_path} — Skipping")

    print(f"✅ Merged '{source1}' and '{source2}' under '{destination}'")
create_merged_structure(
    "/mnt/data/fake_align",
    "/mnt/data/real_align",
    "/mnt/data/merged_align"
)


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random

# --- ISTVT Model (from your code) ---
class ISTVT_TokenPreprocessor(nn.Module):
    def __init__(self, num_frames, num_patches, embed_dim):
        super().__init__()
        self.T, self.N, self.C = num_frames, num_patches, embed_dim
        self.spatial_cls_token = nn.Parameter(torch.randn(1, 1, self.C))
        self.temporal_cls_token = nn.Parameter(torch.randn(1, 1, self.C))
        self.positional_embedding = nn.Parameter(torch.randn(self.T + 1, self.N + 1, self.C))

    def forward(self, patch_tokens):
        T, N, C = patch_tokens.shape
        assert (T, N, C) == (self.T, self.N, self.C)

        spatial_cls = self.spatial_cls_token.expand(T, -1, -1)
        tokens_w_spatial = torch.cat([spatial_cls, patch_tokens], dim=1)
        temporal_cls = self.temporal_cls_token.expand(1, self.N + 1, -1)
        full_sequence = torch.cat([temporal_cls, tokens_w_spatial], dim=0)
        return full_sequence + self.positional_embedding

class SpatialTemporalAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.temporal_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.spatial_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        x_t = x.transpose(0, 1)  # [N+1, T+1, C]
        x = x + self.temporal_attn(self.norm1(x_t), self.norm1(x_t), self.norm1(x_t))[0]
        x_s = x.transpose(0, 1)
        x = x_s + self.spatial_attn(self.norm2(x_s), self.norm2(x_s), self.norm2(x_s))[0]
        return x + self.mlp(x)

class ISTVT_Transformer(nn.Module):
    def __init__(self, num_layers, embed_dim, num_heads):
        super().__init__()
        self.blocks = nn.ModuleList([
            SpatialTemporalAttentionBlock(embed_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x

class PredictionHead(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, 1)
        )

    def forward(self, cls_token):
        x = self.norm(cls_token)
        return self.mlp(x).squeeze()

class ISTVT(nn.Module):
    def __init__(self, num_frames, num_patches, embed_dim, num_heads=8, num_layers=5):
        super().__init__()
        self.preprocessor = ISTVT_TokenPreprocessor(num_frames, num_patches, embed_dim)
        self.transformer = ISTVT_Transformer(num_layers, embed_dim, num_heads)
        self.prediction_head = PredictionHead(embed_dim)

    def forward(self, x):
        x = self.preprocessor(x)
        x = self.transformer(x)
        cls_token = x[0, 0, :]
        return self.prediction_head(cls_token)

# --- Dataset loader for your `processed_clips/` ---
class ClipDataset(Dataset):
    def __init__(self, base_folder):
        self.samples = []
        for label_name, lbl in [('fake', 0), ('real', 1)]:
            folder = os.path.join(base_folder, label_name)
            if not os.path.isdir(folder):
                continue
            for f in os.listdir(folder):
                if f.endswith('.pt'):
                    self.samples.append((os.path.join(folder, f), lbl))
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        tokens = torch.load(path)['tokens']  # [T, N, C]
        return tokens, torch.tensor(lbl, dtype=torch.float32)

# --- Training config and loop ---
def save_checkpoint(model, optimizer, ep, bi, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, f"istvt_epoch{ep}_batch{bi}.pt")
    torch.save({
        'epoch': ep, 'batch': bi,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, path)
    print(f"✅ Saved checkpoint: {path}")

def train(model, dataloader, optimizer, criterion, device, epochs=5, save_every=10):
    model.train()
    for ep in range(1, epochs+1):
        for bi, (clips, labels) in enumerate(dataloader, start=1):
            clips = clips.to(device)  # [B, T, N, C]
            labels = labels.to(device)
            outputs = torch.stack([model(clips[i]) for i in range(clips.size(0))])
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(f"[Epoch {ep}] Batch {bi}, Loss = {loss.item():.4f}")
            if bi % save_every == 0:
                save_checkpoint(model, optimizer, ep, bi, "./checkpoints")

# --- MAIN ENTRYPOINT ---
if __name__ == "__main__":
    # Hyperparams
    T, H, W = 8, 19, 19
    N, C = H*W, 728
    BATCH_SIZE, LR = 4, 1e-4

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ISTVT(num_frames=T, num_patches=N, embed_dim=C).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    dataset = ClipDataset("processed_clips")
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    train(model, dataloader, optimizer, criterion, device)
